# SozoGraph

Portable JSON memory for LLM agents. No vector database, no embedding model,
no local weights.

This notebook builds a memory from a conversation, queries it, moves it through
a file, and shows what the deduplication tiers refuse to do.


## 1. Install

Core is pydantic. Pick whichever engine you have a key for.


In [ ]:
%pip install -q 'sozograph[openai]'
# or: sozograph[anthropic] / sozograph[gemini] / sozograph[ollama]


In [ ]:
import os, getpass

if not os.getenv('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OPENAI_API_KEY: ')


## 2. Build a memory

Turns are batched into token-bounded segments, one extraction call each.
`plan()` tells you the cost before you spend it.


In [ ]:
from sozograph import SozoGraph

conversation = [
    {'speaker': 'Melanie', 'text': 'I finally finished renovating my kitchen.',
     'ts': '2026-01-05T10:00:00Z', 'session': '1'},
    {'speaker': 'Caroline', 'text': 'What colour did you go with?',
     'ts': '2026-01-05T10:01:00Z', 'session': '1'},
    {'speaker': 'Melanie', 'text': "Sage green. I hung my grandmother's painting above the stove.",
     'ts': '2026-01-05T10:02:00Z', 'session': '1'},
    {'speaker': 'Melanie', 'text': 'I live in Harare and I prefer short, direct answers.',
     'ts': '2026-02-10T09:00:00Z', 'session': '2'},
    {'speaker': 'Melanie', 'text': 'I moved to Kwekwe last week for a new job.',
     'ts': '2026-03-11T09:00:00Z', 'session': '3'},
]

sg = SozoGraph('openai:gpt-4o-mini')
sg.plan(conversation)


In [ ]:
passport = sg.ingest(conversation, meta={'user_key': 'melanie'})

print('calls  ', sg.usage.calls)
print('tokens ', sg.usage.total_tokens)
print('facts  ', len(passport.facts))
print('episodes', len(passport.episodes))


## 3. What it remembers

Facts are what is true now. Episodes are what happened, and when.


In [ ]:
for f in passport.facts:
    print(f'{f.key:20s} {f.value}   (confidence {f.confidence})')
print()
for p in passport.prefs:
    print(f'{p.key:20s} {p.value}')
print()
for e in passport.episodes:
    print(f'[{e.ts.date()}] {e.summary}')


### Changes are recorded, not discarded

Melanie moved from Harare to Kwekwe. The newest value wins and the change
is kept, so you can see what your agent used to believe.


In [ ]:
for c in passport.contradictions:
    print(f'{c.key}: {c.old} -> {c.new}')


## 4. Reading it

Facts and preferences go in whole. A query only reorders episodes, so a
retrieval miss can never hide a fact.


In [ ]:
print(passport.context(budget_chars=1200))


In [ ]:
print(passport.context(query='What did she hang above the stove?', budget_chars=800))


## 5. Moving it

The memory is a small JSON file. Copy it anywhere.


In [ ]:
passport.save('melanie.json')

import os
print(os.path.getsize('melanie.json'), 'bytes')
print(passport.token_estimate(), 'tokens')


In [ ]:
from sozograph import Passport

# No key, no SDK, no network needed to read it back.
reloaded = Passport.load('melanie.json')
assert reloaded.to_compact_dict() == passport.to_compact_dict()
print('round trip is lossless')


In [ ]:
import json
print(json.dumps(passport.to_compact_dict()['facts'], indent=2))


## 6. Deduplication

The obvious rule is dangerous. A 0.85 similarity threshold merges
`budget_min` with `budget_max`, and a false merge destroys a real belief
with no undo. Tier 2 refuses on a polarity conflict regardless of score.


In [ ]:
from sozograph.dedupe import compare, jaro_winkler

pairs = [('budget_min', 'budget_max'), ('is_enabled', 'is_disabled'),
         ('has_access', 'has_no_access'), ('likes_coffee', 'likes_toffee'),
         ('Detail Level', 'detail_level'), ('user_location', 'location_user')]

for a, b in pairs:
    m = compare(a, b)
    print(f'{a:14s} vs {b:16s} similarity {jaro_winkler(a, b):.3f}  '
          f'{m.verdict.value:9s} merge={m.is_merge}')


### Tier 3: what string distance cannot reach

`code_style: minimal` and `boilerplate_preference: low` are the same belief.
String similarity is 0.51, so no threshold finds them. One offline call over
the key list does.


In [ ]:
from sozograph import compact

result = compact(passport)
print(result)
print(passport.meta.get('dedupe', {}))


## 7. Any provider

One string changes the engine. Structured output uses each engine's native
mechanism, never a schema pasted into a prompt.


In [ ]:
# SozoGraph('anthropic')                      # forced tool call, strict schema
# SozoGraph('openai:gpt-4o-mini')             # response_format json_schema
# SozoGraph('gemini:gemini-2.5-flash')        # native response_schema
# SozoGraph('ollama:llama3.2')                # grammar-constrained, no key
# SozoGraph('litellm:bedrock/anthropic.claude-opus-5')

from sozograph.providers import DEFAULT_MODELS
DEFAULT_MODELS
